# Transformer

### 1. Using nn.Transformer for the Full Model


We will implement the full Transformer model using PyTorch's built-in `nn.Transformer`, which includes encoder and decoder layers optimized for Transformer architecture. This approach is more efficient and avoids the need to implement individual layers manually.

**Explanation**:
- **nn.Transformer**: Includes all the core components of a Transformer model, including multi-head attention, encoder and decoder layers, and residual connections.
- **nn.Embedding and Positional Encoding**: We use separate embedding and positional encoding modules, as these are not directly included in `nn.Transformer`.

This model can be used for tasks like translation, summarization, and other sequence-to-sequence tasks.
    

In [7]:

import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, embed_size)
        for pos in range(max_len):
            for i in range(0, embed_size, 2):
                pe[pos, i] = math.sin(pos / (10000 ** ((2 * i) / embed_size)))
                pe[pos, i + 1] = math.cos(pos / (10000 ** ((2 * i) / embed_size)))
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, embed_size, num_heads, num_layers, forward_expansion, dropout, max_length):
        super(TransformerModel, self).__init__()
        self.src_embedding = nn.Embedding(src_vocab_size, embed_size)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, embed_size)
        self.positional_encoding = PositionalEncoding(embed_size, max_len=max_length)
        
        # Transformer module with encoder and decoder layers
        self.transformer = nn.Transformer(
            d_model=embed_size, 
            nhead=num_heads, 
            num_encoder_layers=num_layers, 
            num_decoder_layers=num_layers, 
            dim_feedforward=forward_expansion * embed_size, 
            dropout=dropout,
            batch_first=True
        )
        
        self.fc_out = nn.Linear(embed_size, tgt_vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # Embedding and positional encoding for source and target
        src_embedded = self.positional_encoding(self.src_embedding(src))
        tgt_embedded = self.positional_encoding(self.tgt_embedding(tgt))
        
        # Transformer forward pass
        transformer_output = self.transformer(
            src_embedded, tgt_embedded, src_key_padding_mask=src_mask, tgt_key_padding_mask=tgt_mask
        )
        
        # Final output layer for token predictions
        out = self.fc_out(transformer_output)
        return out

# Example usage
src_vocab_size = 10000
tgt_vocab_size = 10000
embed_size = 16
num_heads = 4
num_layers = 3
forward_expansion = 4
dropout = 0.1
max_length = 100

transformer = TransformerModel(src_vocab_size, tgt_vocab_size, embed_size, num_heads, num_layers, forward_expansion, dropout, max_length)
src = torch.randint(0, src_vocab_size, (2, 5))  # Batch size = 2, Sequence length = 5
print("Source Input:", src)
tgt = torch.randint(0, tgt_vocab_size, (2, 5))
print("Target Input:", tgt)

output = transformer(src, tgt)
print("Transformer Model Output (logits):", output)
    

Source Input: tensor([[7032, 8626, 6591, 6003, 1479],
        [4563, 4392, 7484, 9082, 1101]])
Target Input: tensor([[6322, 3611, 8801, 8466,  335],
        [3945, 9967, 9036, 8229, 9931]])
Transformer Model Output (logits): tensor([[[ 0.0912, -0.2319, -0.4459,  ...,  0.1217,  0.3181, -0.3905],
         [ 0.2395,  0.0942,  0.0779,  ...,  0.6076, -0.2470, -0.2912],
         [-0.2071, -0.0396,  0.0632,  ..., -0.2792,  0.4937, -0.8662],
         [ 0.5084, -0.1963,  0.3725,  ...,  0.1915,  0.1093, -0.8208],
         [ 0.2553, -0.1949,  0.1512,  ...,  0.4509, -0.1293, -0.5978]],

        [[ 0.4755, -0.0872, -0.0658,  ...,  0.5186,  0.0226, -0.6906],
         [ 0.1987,  0.0499, -0.0086,  ...,  0.6053, -0.3479, -0.3830],
         [ 0.4156, -0.0809, -0.0811,  ...,  0.6174, -0.1172, -0.6645],
         [-0.0931,  0.3523, -0.2325,  ...,  0.0942, -0.0330, -0.6530],
         [ 0.2531,  0.1023, -0.1798,  ...,  0.4872, -0.2815, -0.1683]]],
       grad_fn=<ViewBackward0>)


### 2. Training Loop Example


**Objective**: Demonstrate a basic training loop with a forward and backward pass, typically used to train the Transformer on sequence-to-sequence tasks.

**Explanation**:
   - **Loss Calculation**: The model output is compared to the target sequence to compute cross-entropy loss.
   - **Backpropagation**: The loss is used to adjust model weights through gradient descent.
   - **Optimization**: Each batch adjusts model parameters to improve predictions.
    

In [8]:
import torch.optim as optim

# Example training loop
num_epochs = 10
learning_rate = 0.001
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Assuming padding index is 0
optimizer = optim.Adam(transformer.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    transformer.train()
    optimizer.zero_grad()

    output = transformer(src, tgt[:, :-1])  # Input all tokens except the last one
    # [batch_size, sequence_length, vocab_size]
    output = output.reshape(
        -1, output.shape[2]
    )  # [batch_size * sequence_length, vocab_size]
    tgt_output = tgt[:, 1:].reshape(-1)  # Expected output sequence shifted by one
    # tgt[:, 1:]: The target sequence, excluding the first token. 
    # This is done because the model's prediction at each time step should be compared to the next token in the sequence.
    # tgt[:, 1:].reshape(-1): Reshapes the target sequence to a 1D tensor of shape [batch_size * sequence_length]. 
    # This flattening aligns with the reshaped model output for loss computation.
    
    loss = criterion(output, tgt_output)
    loss.backward()
    optimizer.step()

    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [1/10], Loss: 8.9757
Epoch [2/10], Loss: 8.8692
Epoch [3/10], Loss: 8.8674
Epoch [4/10], Loss: 8.5676
Epoch [5/10], Loss: 8.6081
Epoch [6/10], Loss: 8.5902
Epoch [7/10], Loss: 8.4101
Epoch [8/10], Loss: 8.4268
Epoch [9/10], Loss: 8.4158
Epoch [10/10], Loss: 8.2947
